In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn

In [2]:

from fsd50k_setup import ( 
    train_loader, val_loader, eval_loader, 
    NUM_CLASSES, decode_labels, 
    FRAME_COUNT
)

# Sanity check
features, labels = next(iter(train_loader))
print(f"Feature batch shape : {features.shape}")
print(f"Label batch shape   : {labels.shape}")
print(f"First clip labels   : {decode_labels(labels[0])}")

Feature batch shape : torch.Size([8, 1, 128, 516])
Label batch shape   : torch.Size([8, 200])
First clip labels   : ['Crumpling_and_crinkling']


### Build model

In [3]:
class AudioCNN(nn.Module):
    """
    Four convolutional blocks followed by global average pooling
    and a linear classifier.

    Each block:
        Conv2d → BatchNorm → ReLU → MaxPool """


    def __init__(self, num_classes: int = 200):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1 — learns low-level features (edges in freq/time)
            # Input:  (batch, 1,   128, 516)
            # Output: (batch, 32,  64,  258)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),              # halves both dimensions

            # Block 2 — learns mid-level patterns (harmonics, rhythms)
            # Input:  (batch, 32,  64,  258)
            # Output: (batch, 64,  32,  129)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3 — learns higher-level combinations
            # Input:  (batch, 64,  32,  129)
            # Output: (batch, 128, 16,  64)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 4 — learns abstract sound representations
            # Input:  (batch, 128, 16,  64)
            # Output: (batch, 128, 8,   32)
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        # Global average pooling: (batch, 128, 8, 32) → (batch, 128)
        self.gap = nn.AdaptiveAvgPool2d(1)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),              # regularization — randomly zeroes 50% of inputs
            nn.Linear(128, num_classes),  # (batch, 128) → (batch, 200)
            # No sigmoid here — BCEWithLogitsLoss applies it internally,
            # which is numerically more stable
        )

    def forward(self, x):
        x = self.features(x)    # (batch, 128, 8, 32)
        x = self.gap(x)         # (batch, 128, 1, 1)
        x = x.flatten(1)        # (batch, 128)
        x = self.classifier(x)  # (batch, 200)
        return x

### Training

In [4]:

# ─────────────────────────────────────────────
# TRAINING SETUP
# ─────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model     = AudioCNN(num_classes=NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()   # correct loss for multi-label classification

# Quick parameter count
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")


# ─────────────────────────────────────────────
# TRAINING AND VALIDATION FUNCTIONS
# ─────────────────────────────────────────────
def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    for features, labels in loader:
        features = features.to(device, non_blocking=True)
        labels   = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(features)          # (batch, 200) raw logits
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(loader):
    """
    Returns average loss and per-sample accuracy.
    Accuracy here = fraction of the 200 labels predicted correctly
    per clip (both 0s and 1s), averaged across all clips.
    This is called 'subset accuracy' or 'exact match' in multi-label literature.
    """
    model.eval()
    total_loss     = 0.0
    total_correct  = 0
    total_elements = 0

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device, non_blocking=True)
            labels   = labels.to(device, non_blocking=True)

            logits = model(features)
            loss   = criterion(logits, labels)
            total_loss += loss.item()

            # Convert logits to binary predictions at threshold 0.5
            preds   = (torch.sigmoid(logits) > 0.5).float()
            correct = (preds == labels).sum().item()
            total_correct  += correct
            total_elements += labels.numel()

    avg_loss     = total_loss / len(loader)
    avg_accuracy = total_correct / total_elements
    return avg_loss, avg_accuracy

Using device: cpu
Trainable parameters: 266,760


In [5]:
# ─────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────
NUM_EPOCHS = 10

print(f"\nTraining for {NUM_EPOCHS} epochs...\n")
print(f"{'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14} {'Val Accuracy'}")
print("-" * 50)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss          = train_one_epoch(train_loader)
    val_loss, val_acc   = evaluate(val_loader)

    print(f"{epoch:<8} {train_loss:<14.4f} {val_loss:<14.4f} {val_acc:.4f}")


Training for 10 epochs...

Epoch    Train Loss     Val Loss       Val Accuracy
--------------------------------------------------


KeyboardInterrupt: 

In [5]:
import time

# Time just ONE batch through the model
features, labels = next(iter(train_loader))
features = features.to(device)
labels   = labels.to(device)

start = time.time()
with torch.no_grad():
    logits = model(features)
print(f"One forward pass: {time.time()-start:.2f}s")
print(f"Batches per epoch: {len(train_loader)}")
print(f"Estimated epoch time: {(time.time()-start) * len(train_loader) / 60:.1f} minutes")

One forward pass: 0.60s
Batches per epoch: 250
Estimated epoch time: 2.5 minutes


In [6]:
# Check 1 — are we actually on CPU?
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Check 2 — how much RAM is being used?
import psutil
ram = psutil.virtual_memory()
print(f"RAM used: {ram.used/1e9:.1f} GB / {ram.total/1e9:.1f} GB")
print(f"RAM available: {ram.available/1e9:.1f} GB")

# Check 3 — how loaded is the CPU during a forward pass?
import time
start = time.time()
for i, (features, labels) in enumerate(train_loader):
    features = features.to(device)
    labels = labels.to(device)
    logits = model(features)
    if i == 5:
        break
print(f"5 batches took: {time.time()-start:.2f}s")

Device: cpu
CUDA available: False
RAM used: 4.0 GB / 6.4 GB
RAM available: 2.4 GB
5 batches took: 4.40s
